# Machine Learning > Supervised Learning > Regression 
## The first set of examples can be found in: regression_examples.ipynb
## Two more examples: Random Forest and Gaussian Process

In [ ]:
# Import the needed packages for math and plotting
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random

# Random Forest packages from scikit-learn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Gaussian Process packages from scikit-learn
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

In [ ]:
# RANDOM FOREST
# Choose the number of trees = 'n'
n  = int(input('Choose a number for n: '))

# Generate sample data 
rng = np.random.RandomState(1)
X = np.sort(5 * rng.rand(200, 1), axis=0)
y = np.sin(X).ravel()
y[::5] += 3 * (0.5 - rng.rand(40))

# Split into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the model and call it "rf"
rf = RandomForestRegressor(
    n_estimators=n,  # number of decision trees
    max_depth=10,  # Limit tree depth to prevent overfitting
    n_jobs=-1,  # Use all CPU cores for speed
    random_state=42)  # Fixed seed for reproducibility)

# Train the model
rf.fit(X_train, y_train)

# Make predictions
predictions = rf.predict(X_test)

# 6. Evaluate accuracy
mse = mean_squared_error(y_test, predictions)
r2_score = rf.score(X_test, y_test)

print(f"R-squared Score: {r2_score:.4f}")

# plot the model together with the data
plt.figure()
plt.scatter(X_train, y_train, c="darkorange", alpha=0.6, label="Train Data")
plt.scatter(X_test, y_test, c="crimson", marker="x", label="Test Data")
plt.scatter(X_test, predictions, color="cornflowerblue", label="# Trees ="+str(n), s=50, marker='s')
plt.xlabel("data")
plt.ylabel("target")
plt.title("Random Forest")
plt.legend()
plt.show()

In [ ]:
# GAUSSIAN PROCESS
# Choose the length scale of the Radial Bias Function (RBF) = 'l'
l = float(input("Choose Radial Bias Function length scale (e.g., 0.1, 1.0, 5.0): "))

# Generate sample data 
rng = np.random.RandomState(1)
X = np.sort(5 * rng.rand(200, 1), axis=0)
y = np.sin(X).ravel()
y[::5] += 3 * (0.5 - rng.rand(40))

# Split into training (80%) and test (20%) sets
# Train/Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define Kernel with fixed length scale so optimizer won't override user choice
kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=l, length_scale_bounds="fixed")

# Initialize & Fit Gaussian Process Regressor
gp = GaussianProcessRegressor(kernel=kernel, alpha=0.1, random_state=42)
gp.fit(X_train, y_train)

# Evaluate on Test Set
y_test_pred = gp.predict(X_test)
r2 = gp.score(X_test, y_test)

print(f"R-squared Score:    {r2:.4f}")

# Generate smooth curve and uncertainty estimates for plotting
X_grid = np.linspace(0, 5, 500).reshape(-1, 1)
y_grid_pred, std = gp.predict(X_grid, return_std=True)

# Plot Results
plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, c="darkorange", alpha=0.6, label="Train Data")
plt.scatter(X_test, y_test, c="crimson", marker="x", label="Test Data")
plt.plot(X_grid, y_grid_pred, color="cornflowerblue", linewidth=2.5, label=f"GP Mean (l={l})")
# 95% Confidence Interval (± 1.96 * std_dev)
plt.fill_between(
    X_grid.ravel(),
    y_grid_pred - 1.96 * std,
    y_grid_pred + 1.96 * std,
    color="cornflowerblue",
    alpha=0.2,
    label="95% Confidence Interval")
plt.xlabel("X")
plt.ylabel("y")
plt.title(f"Gaussian Process Regression (RBF Length Scale = {l})")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()